In [1]:
#Import bibliotecas
import os
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, count, col, current_timestamp, from_json, lit, row_number, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
#Iniciando sessão spark
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9cef5a7c-88e4-4d6c-9d75-a1c47f5f9b6a;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 224ms :: artifacts dl 9ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number

In [5]:
!ls /tmp/warehouse/bronze/transacoes

data  metadata


In [6]:
#Variaveis de conexao ao ambiente postgres
conexao = {
           "url": "jdbc:postgresql://postgres:5432/pipeline_transacoes",
           "user": "postgres",
           "password": "1234",
           "driver": "org.postgresql.Driver"
}

In [7]:
#Dados vindo da postgres
df_raw = spark.read.jdbc(
    url=conexao["url"],
    table="staging.transacoes_raw",  # tabela no PostgreSQL
    properties={
        "user": conexao["user"],
        "password": conexao["password"],
        "driver": conexao["driver"]
    }
)

df_raw.select("payload").show(truncate=True)

+-------+
|payload|
+-------+
+-------+



In [8]:
#Criando tabela bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

#Definir schema do payload
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("metodo_pagamento", StringType(), True),
])

# Transformar DF raw
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_particao", to_date(col("data_ingestao")))
    .withColumn("payload_struct", from_json(col("payload"), schema_payload))
    .select(
        "id",
        "payload_struct.*",
        "data_ingestao",
        "data_particao"
    )
)                           
df_bronze.writeTo("lakehouse.bronze.transacoes") \
         .using("iceberg") \
         .partitionedBy("data_particao") \
         .append()
    

In [28]:
spark._sc._jvm.java.sql.DriverManager.getConnection(
    conexao["url"],
    conexao["user"],
    conexao["password"]
).createStatement().execute("TRUNCATE TABLE staging.transacoes_raw")

False

In [29]:
df_raw = spark.read \
    .format("jdbc") \
    .options(**conexao) \
    .option("dbtable", "transacoes") \
    .load()

Py4JJavaError: An error occurred while calling o257.load.
: org.postgresql.util.PSQLException: ERROR: relation "transacoes" does not exist
  Position: 15
	at org.postgresql.core.v3.QueryExecutorImpl.receiveErrorResponse(QueryExecutorImpl.java:2713)
	at org.postgresql.core.v3.QueryExecutorImpl.processResults(QueryExecutorImpl.java:2401)
	at org.postgresql.core.v3.QueryExecutorImpl.execute(QueryExecutorImpl.java:368)
	at org.postgresql.jdbc.PgStatement.executeInternal(PgStatement.java:498)
	at org.postgresql.jdbc.PgStatement.execute(PgStatement.java:415)
	at org.postgresql.jdbc.PgPreparedStatement.executeWithFlags(PgPreparedStatement.java:190)
	at org.postgresql.jdbc.PgPreparedStatement.executeQuery(PgPreparedStatement.java:134)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.getQueryOutputSchema(JDBCRDD.scala:68)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.resolveTable(JDBCRDD.scala:58)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRelation$.getSc

In [ ]:
print(" Total de registros na RAW (Postgres):")

df_raw.count()

In [10]:
#Select tabela transacoes na bronze
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:38:...|   2026-04-12|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:38:...|   2026-04-12|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:38:...|   2026-04-12|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:38:...|   2026-04-12|
|  5|   NULL|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:38:...|   2026-04-12|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+



In [ ]:
spark.sql("""
SELECT data_particao, COUNT(*) as qtd
FROM lakehouse.bronze.transacoes
GROUP BY data_particao
ORDER BY data_particao
""").show()

In [11]:
#Validando estrutura tabela bronze
spark.sql("Describe lakehouse.bronze.transacoes").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|                  id|      int|   NULL|
|               valor|   double|   NULL|
|             produto|   string|   NULL|
|           categoria|   string|   NULL|
|          cliente_id|      int|   NULL|
|          quantidade|      int|   NULL|
|    metodo_pagamento|   string|   NULL|
|       data_ingestao|timestamp|   NULL|
|       data_particao|     date|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|       data_particao|     date|   NULL|
+--------------------+---------+-------+



In [12]:
#Validando a existencia de dados nulos na bronze

df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_bronze.columns
]).show()

+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|  id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|NULL| NULL|   NULL|     NULL|      NULL|      NULL|            NULL|         NULL|         NULL|
+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [13]:
#Verificando dados nulos no campo VALOR
spark.sql("SELECT * FROM lakehouse.bronze.transacoes where valor is null").show()

+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  5| NULL|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:38:...|   2026-04-12|
| 19| NULL|    fone|eletronicos|        74|         2|          boleto|2026-04-12 17:38:...|   2026-04-12|
| 23| NULL|notebook|eletronicos|         6|         1|             pix|2026-04-12 17:38:...|   2026-04-12|
| 27| NULL|camiseta|  vestuario|        32|         1|          boleto|2026-04-12 17:38:...|   2026-04-12|
| 31| NULL| celular|eletronicos|        16|         3|             pix|2026-04-12 17:38:...|   2026-04-12|
| 36| NULL|notebook|eletronicos|        97|         1|          boleto|2026-04-12 17:38:...|   2026-04-12|
| 40| NULL| celular|eletronicos|     

In [14]:
#Validando se a duplicata para possivel tratamento
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

df_bronze.groupBy("id", "data_ingestao") \
    .agg(count("*").alias("qtd")) \
    .filter(col("qtd") > 1) \
    .show()

+---+-------------+---+
| id|data_ingestao|qtd|
+---+-------------+---+
+---+-------------+---+



In [15]:
#  TRANSFORMAÇÕES (SILVER)
# - tratamento de null
# - deduplicação
window = Window.partitionBy("id").orderBy(col("data_ingestao").desc())

df_curated = (
    df_bronze
    .fillna({"valor": 0})
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .withColumn("preco_unitario", col("valor") / col("quantidade"))
)

# Criar database
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

# Salvar
df_curated.writeTo("lakehouse.silver.transacoes") \
          .using("iceberg") \
          .partitionedBy("data_particao") \
          .createOrReplace()

spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:42:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:42:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:42:...|   2026-04

In [16]:
spark.sql("""
SELECT id, COUNT(*) 
FROM lakehouse.silver.transacoes
GROUP BY id
HAVING COUNT(*) > 1
""").show()

+---+--------+
| id|count(1)|
+---+--------+
+---+--------+



In [17]:
df_curated.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [18]:
print(" Validando duplicatas após tratamento...")

df_curated.groupBy("id", "data_ingestao") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print(" Dados finais na Silver:")
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

 Validando duplicatas após tratamento...
+---+-------------+-----+
| id|data_ingestao|count|
+---+-------------+-----+
+---+-------------+-----+

 Dados finais na Silver:
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:42:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:42:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|     

In [19]:
#Atualização na tabela silver (ADD COLUMNS)
df_add = df_curated.withColumn("preco_unitario", (col("valor") / col("quantidade")))
df_add.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [20]:
#Validando após alteração
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:42:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:42:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:42:...|   2026-04

In [21]:
!ls /tmp/warehouse/bronze/transacoes/data/data_particao=2026-04-12

00000-2-0d1f9665-96ab-4c13-9e23-f70ba73cea92-00001.parquet
00000-2-19734366-bc65-4857-938c-d120b2c0784d-00001.parquet
00000-2-4079e7d3-717a-462e-8a05-7615e9019dbb-00001.parquet
00000-2-4e545517-cf8c-43d5-990a-0ce890edfe70-00001.parquet
00000-2-4ead8a51-f7cd-4401-9723-14a458d5301e-00001.parquet
00000-2-57ff789e-e4de-4480-8b25-bdd07844e6a5-00001.parquet
00000-2-6c7cc06c-9cfc-4b99-85de-436552f31d11-00001.parquet
00000-2-9d46c383-2613-464a-add0-5b34a5333312-00001.parquet
00000-2-a5b212e8-fd79-4b69-ae76-fcec62843d66-00001.parquet
00000-2-a924c076-98c1-4f89-936a-ffde7769d073-00001.parquet
00000-2-d84e648f-c7ca-4e03-b83d-61dcd13458ed-00001.parquet
00000-2-e1b5390c-fce2-4f2e-a324-74857c87552a-00001.parquet
00000-2-ea6d4989-dd19-4bd0-bab7-ff2f3e4c1cf1-00001.parquet
00000-8-8396e66f-e1db-4ba4-8e7f-7f62199a6118-00001.parquet
00000-9-19d438e3-7eb4-498a-abe7-069fa9113305-00001.parquet


In [22]:
print(" ANTES do DELETE:")

df_antes = spark.sql("""
SELECT * FROM lakehouse.silver.transacoes
WHERE id = 1
""")

df_antes.show()

 ANTES do DELETE:
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+--------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+--------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|       1218.61|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+--------------+



In [23]:
print("Executando DELETE...")

spark.sql("""
DELETE FROM lakehouse.silver.transacoes
WHERE id = 1
""")

spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

Executando DELETE...
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:42:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:42:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:42:...|   2026-04-12|               0.0|
|  6|1942.65|notebook|eletronicos|        15|         1|             pix|2026-04-12

In [24]:
print(" Snapshots da tabela:")

spark.sql("""
SELECT 
  snapshot_id,
  parent_id,
  committed_at,
  operation
FROM lakehouse.silver.transacoes.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

 Snapshots da tabela:
+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|6886291932207073309|8171409476006962084|2026-04-12 17:44:38.81 |overwrite|
|8171409476006962084|NULL               |2026-04-12 17:44:36.919|append   |
|6341660002184003658|NULL               |2026-04-12 17:44:35.466|append   |
|5966603098831202129|NULL               |2026-04-12 17:44:33.191|append   |
|8258277374760095342|4945707606062990547|2026-04-12 17:35:02.014|overwrite|
|4945707606062990547|NULL               |2026-04-12 17:34:59.747|append   |
|636509152785229609 |NULL               |2026-04-12 17:34:58.004|append   |
|5181906299346480431|NULL               |2026-04-12 17:34:56.78 |append   |
|1876818666659084853|1662487065579877806|2026-04-12 17:28:21.804|overwrite|
|1662487065579877806|NULL               |2026-04-12 17:28:20.068|a

In [25]:
print(" Time Travel...")

snapshots = spark.sql("""
SELECT snapshot_id, committed_at
FROM lakehouse.silver.transacoes.snapshots
ORDER BY committed_at DESC
""").collect()

if len(snapshots) > 1:
    snapshot_id = snapshots[1]["snapshot_id"]
    
    print(f" Voltando para snapshot: {snapshot_id}")

    spark.sql(f"""
    SELECT * FROM lakehouse.silver.transacoes
    VERSION AS OF {snapshot_id}
    """).show()
else:
    print(" Não há snapshots suficientes para Time Travel")

 Time Travel...
 Voltando para snapshot: 8171409476006962084
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:42:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:42:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:42:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        7

In [26]:
df_test = spark.range(0, 1000000)

print(" Comparando performance...")

# PARQUET
start = time.time()
df_test.write.mode("overwrite").parquet("/tmp/parquet_test")
print(f" Escrita Parquet: {time.time() - start:.4f}s")

start = time.time()
spark.read.parquet("/tmp/parquet_test").count()
print(f" Leitura Parquet: {time.time() - start:.4f}s")

# ICEBERG
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.silver.perf_test (
    id BIGINT
)
USING iceberg
""")

start = time.time()
df_test.writeTo("lakehouse.silver.perf_test").overwritePartitions()
print(f" Escrita Iceberg: {time.time() - start:.4f}s")

start = time.time()
spark.read.table("lakehouse.silver.perf_test").count()
print(f" Leitura Iceberg: {time.time() - start:.4f}s")

 Comparando performance...


 Escrita Parquet: 2.1835s
 Leitura Parquet: 0.7677s


 Escrita Iceberg: 1.1565s
 Leitura Iceberg: 0.1882s
